# 0. Imports

In [ ]:
import numpy as np
import pandas as pd


import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor


from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 50)
PAX_TARGET = "OD_PAX_LOG"  
REV_TARGET = "OD_REV_USD_LOG"      
VAL_MONTHS = 12             


# 1. Organizing

In [20]:
full_spine = pd.read_csv('./data/made/AA_MASTER_V10.csv')

C:\Users\danie\AppData\Local\Temp\ipykernel_19896\1665662883.py:1: DtypeWarning: Columns (14,15,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  full_spine = pd.read_csv('./data/made/AA_MASTER_V10.csv')


In [21]:
full_spine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1968685 entries, 0 to 1968684
Data columns (total 61 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   ORG_AIRPRT               object 
 1   DES_AIRPRT               object 
 2   DEP_YEAR                 int64  
 3   DEP_MONTH                int64  
 4   ROUTE                    object 
 5   OD_REV_USD_LOG           float64
 6   OD_PAX_LOG               float64
 7   WEEKEND                  int64  
 8   HOLIDAY                  int64  
 9   EASTER                   int64  
 10  COVID                    int64  
 11  MPAX_L24                 float64
 12  PAX_SHARE_L24            float64
 13  AIRLN_COUNT_L24          float64
 14  ORG_CARGO_SL             object 
 15  ORG_CARGO_HUB            object 
 16  DES_CARGO_SL             object 
 17  DES_CARGO_HUB            object 
 18  ORG_ENP_SL               object 
 19  ORG_ENP_HUB              object 
 20  DES_ENP_SL               object 
 21  DES_ENP_

In [ ]:
full_spine.drop(columns=['ORG_CARGO_SL','ORG_CARGO_HUB','DES_CARGO_SL','DES_CARGO_HUB','ORG_CNTRY_CD',
                         'DES_CNTRY_CD'],inplace=True)
full_spine.describe()


,DEP_YEAR,DEP_MONTH,OD_REV_USD_LOG,OD_PAX_LOG,WEEKEND,HOLIDAY,EASTER,COVID,MPAX_L24,PAX_SHARE_L24,AIRLN_COUNT_L24,OD_PAX_LOG_L24,OD_REV_USD_LOG_L24,L24_COVID,ORG_LNGST_RUNWAY_FT_QTY,ORG_ELEVATN_FT_QTY,DES_LNGST_RUNWAY_FT_QTY,DES_ELEVATN_FT_QTY,ORG_CNTY_GDP_LOG,ORG_CNTY_INC,DES_CNTY_GDP_LOG,DES_CNTY_INC,ORG_CNTRY_GDP,DES_CNTRY_GDP,ORG_CBSA_POP,ORG_CBSA_POP_GRWT,ORG_CBSA_NET_MIG,DES_CBSA_POP,DES_CBSA_POP_GRWT,DES_CBSA_NET_MIG,MIG_OUT_LOG,MIG_IN_LOG,MIG_GROSS_LOG,MIG_NET_LOG,DIST_LOG,ORG_CNTY_INC_LOG,DES_CNTY_INC_LOG,INTRNAT,ECON_LOG,ECON_SUM_LOG,PURCHASE_PWR_LOG,AVG_FARE_LOG_L24,MONTH_SIN,MONTH_COS,SUMMER,WINTER,COMB_CARGO_WGT,COMB_ENP_LOG
count,1.968685e+06,1.968685e+06,1.557829e+06,1.557829e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.449115e+06,1.449115e+06,1.449115e+06,1.557829e+06,1.557829e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.483119e+06,1.483119e+06,1.496689e+06,1.496689e+06,1.939253e+06,1.939425e+06,1.483845e+06,1.483845e+06,1.483845e+06,1.496725e+06,1.496725e+06,1.496725e+06,1.045350e+06,1.045350e+06,1.045350e+06,1.045350e+06,1.968685e+06,1.483119e+06,1.496689e+06,1.968685e+06,1.909993e+06,1.909993e+06,1.909993e+06,1.533522e+06,1.968685e+06,1.968685e+06,1.968685e+06,1.968685e+06,491538.000000,1.063397e+06
mean,2.023304e+03,6.347826e+00,1.173114e+01,6.118466e+00,8.695652e+00,1.017391e+00,8.695652e-02,2.086957e-01,4.814356e+03,5.079045e-02,1.034191e+00,6.118466e+00,1.173114e+01,2.086957e-01,1.073016e+04,8.883565e+02,1.071401e+04,8.912647e+02,2.487358e+01,6.838344e+04,2.487278e+01,6.835958e+04,1.092147e+01,1.093625e+01,1.450325e+01,6.369078e-03,3.718219e-03,1.450374e+01,6.364739e-03,3.722595e-03,7.862422e-03,9.478168e-03,1.248336e-02,2.113734e-03,7.457973e+00,1.109095e+01,1.109063e+01,4.690110e-01,3.530473e+01,4.275131e+01,2.167184e+01,5.701642e+00,2.810479e-02,-1.622631e-02,2.521739e-01,1.652174e-01,40.911514,3.049126e+01
std,2.773082e+00,3.433958e+00,2.399621e+00,2.148589e+00,6.348423e-01,7.910656e-01,2.817714e-01,4.063765e-01,1.156295e+04,1.974159e-01,1.457856e+00,2.148589e+00,2.399621e+00,4.063765e-01,1.884686e+03,1.495306e+03,1.882790e+03,1.493425e+03,1.055841e+00,2.145534e+04,1.054456e+00,2.144105e+04,6.918877e-01,6.692747e-01,1.116705e+00,1.117748e-02,8.191824e-03,1.113010e+00,1.118023e-02,8.197560e-03,1.733969e-01,1.939653e-01,2.317520e-01,2.013257e-01,9.022118e-01,2.824055e-01,2.822975e-01,4.990389e-01,8.180532e+00,7.642684e+00,9.127765e-01,6.468867e-01,7.050081e-01,7.084567e-01,4.342607e-01,3.713767e-01,2.275107,1.673394e+00
min,2.019000e+03,1.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.700000e+03,-1.100000e+01,5.700000e+03,-1.100000e+01,2.201507e+01,2.630600e+04,2.201507e+01,2.630600e+04,7.553833e+00,7.553833e+00,1.120110e+01,-2.575449e-02,-2.684170e-02,1.120110e+01,-2.575449e-02,-2.684170e-02,0.000000e+00,0.000000e+00,0.000000e+00,-4.955827e+00,3.090996e+00,1.017759e+01,1.017759e+01,0.000000e+00,1.042886e+01,1.902224e+01,1.775374e+01,0.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,30.058599,2.511071e+01
25%,2.021000e+03,3.000000e+00,1.067351e+01,4.788907e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.788907e+00,1.067351e+01,0.000000e+00,9.500000e+03,4.600000e+01,9.500000e+03,4.700000e+01,2.421654e+01,5.429300e+04,2.421654e+01,5.429300e+04,1.100291e+01,1.100291e+01,1.373785e+01,-1.998730e-04,-9.120911e-04,1.373785e+01,-1.998730e-04,-9.120911e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.881366e+00,1.090217e+01,1.090217e+01,0.000000e+00,2.703651e+01,3.502019e+01,2.131474e+01,5.288252e+00,-5.000000e-01,-8.660254e-01,0.000000e+00,0.000000e+00,39.662461,2.925247e+01
50%,2.023000e+03,6.000000e+00,1.192177e+01,6.181754e+00,9.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,6.181754e+00,1.192177e+01,0.000000e+00,1.100000e+04,3.550000

In [25]:
full_spine.to_csv('./data/made/AA_MASTER_V10.csv',index=False)

In [ ]:
def prepare_datasets(
    df: pd.DataFrame,
    target_col: str,
    val_months: int = 12,
    drop_covid_targets: bool = True,
):
    """
    Separates `full_spine` into leak-free train, validation, and 2027-2028 forecast sets.
    Automatically casts object/string features to `category` dtype for native tree support.
    """
    work_df = df.copy()

    # 1. Define columns that dont enter X 
    non_feature_cols = {
        "ROUTE", "ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH",
        PAX_TARGET, REV_TARGET, "COVID",
    }
    feature_cols = [c for c in work_df.columns if c not in non_feature_cols]

    # 2. Cast remaining object/string columns to 'category'
    cat_cols = work_df[feature_cols].select_dtypes(include=["object", "string"]).columns.tolist()
    for col in cat_cols:
        work_df[col] = work_df[col].astype("category")

    # 3. Create helper chronological timestamp for exact time-based slicing
    work_df["_ts"] = pd.to_datetime(
        dict(year=work_df["DEP_YEAR"], month=work_df["DEP_MONTH"], day=1)
    )

    # Auto-detect exact historical cutoff T 
    cutoff_t = work_df.loc[work_df[target_col].notna(), "_ts"].max()
    val_start = cutoff_t - pd.DateOffset(months=val_months - 1)

    print(f"Historical Cutoff T: {cutoff_t.strftime('%Y-%m')}")
    print(f"Validation Window:   {val_start.strftime('%Y-%m')} -> {cutoff_t.strftime('%Y-%m')}")

    # 4. Filter valid historical rows 
    hist_mask = work_df[target_col].notna() & work_df[f"{target_col}_L24"].notna()
    if drop_covid_targets:
        hist_mask = hist_mask & (~work_df["DEP_YEAR"].isin([2020, 2021]))

    train_df = work_df[hist_mask & (work_df["_ts"] < val_start)].copy()
    val_df = work_df[hist_mask & (work_df["_ts"] >= val_start)].copy()
    forecast_df = work_df[work_df["_ts"] > cutoff_t].copy()

    return train_df, val_df, forecast_df, feature_cols, cat_cols


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

def get_tree_models(cat_cols: list, num_cols: list) -> dict:
    """Returns native-NaN gradient boosters + a NaN-safe Random Forest pipeline."""
    
    # Preprocessing for RandomForest 
    rf_preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="constant", fill_value=-999.0), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2), cat_cols),
        ]
    )

    rf_pipeline = Pipeline([
        ("prep", rf_preprocessor),
        ("rf", RandomForestRegressor(
            n_estimators=150,       
            max_depth=16,           
            min_samples_leaf=10,
            max_samples=0.5,        
            random_state=42,
            n_jobs=-1,
        )),
    ])

    return {
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            subsample_for_bin=200_000, random_state=42, n_jobs=-1,
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=7,
            tree_method="hist", enable_categorical=True, random_state=42, n_jobs=-1,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=500, learning_rate=0.05, max_leaf_nodes=63,
            categorical_features="from_dtype", random_state=42,
        ),
        "RandomForest": rf_pipeline,
    }


def evaluate_predictions(y_true_log1p: pd.Series, y_pred_log1p: np.ndarray) -> dict:
    """Evaluates error in BOTH log1p space (model scale) and raw expm1 space (real units)."""
   
    y_pred_log1p = np.clip(y_pred_log1p, 0.0, None)

    y_true_raw = np.expm1(y_true_log1p)
    y_pred_raw = np.expm1(y_pred_log1p)

    return {
        "RMSE_log1p": round(root_mean_squared_error(y_true_log1p, y_pred_log1p), 4),
        "R2_log1p": round(r2_score(y_true_log1p, y_pred_log1p), 4),
        "MAE_raw": round(mean_absolute_error(y_true_raw, y_pred_raw), 2),
        "RMSE_raw": round(root_mean_squared_error(y_true_raw, y_pred_raw), 2),
    }


In [ ]:

train_df, val_df, forecast_df, feature_cols, cat_cols = prepare_datasets(
    df=full_spine,
    target_col=PAX_TARGET,
    val_months=VAL_MONTHS,
    drop_covid_targets=True,
)

Historical Cutoff T: 2026-07
Validation Window:   2025-08 -> 2026-07


In [ ]:

X_train, y_train = train_df[feature_cols], train_df[PAX_TARGET]
X_val, y_val = val_df[feature_cols], val_df[PAX_TARGET]

print(f"Train rows: {len(X_train):,} | Val rows: {len(X_val):,} | Features: {len(feature_cols)}")

# Train & benchmark each tree model
models = get_tree_models(cat_cols, num_cols)
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")  
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    metrics = evaluate_predictions(y_val, val_preds)
    metrics["Model"] = name
    results.append(metrics)

# 3. Compare validation scores side-by-side
results_df = pd.DataFrame(results)[["Model", "RMSE_log1p", "R2_log1p", "MAE_raw", "RMSE_raw"]]
print("\n--- Validation Benchmark ---")
print(results_df.to_string(index=False))


Train rows: 736,117 | Val rows: 205,428 | Features: 47

Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006877 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6495
[LightGBM] [Info] Number of data points in the train set: 736117, number of used features: 47
[LightGBM] [Info] Start training from score 6.481447

Training XGBoost...

Training HistGradientBoosting...

Training RandomForest...

--- Validation Benchmark ---
               Model  RMSE_log1p  R2_log1p  MAE_raw  RMSE_raw
            LightGBM      0.5523    0.9037   774.38   2004.11
             XGBoost      0.5178    0.9154   666.63   1778.23
HistGradientBoosting      0.5947    0.8884  1059.62   2916.44
        RandomForest      0.5320    0.9107   635.88   1543.06


In [28]:

train_df, val_df, forecast_df, feature_cols, cat_cols = prepare_datasets(
    df=full_spine,
    target_col=REV_TARGET,
    val_months=VAL_MONTHS,
    drop_covid_targets=True,
)

Historical Cutoff T: 2026-07
Validation Window:   2025-08 -> 2026-07


In [ ]:

X_train, y_train = train_df[feature_cols], train_df[REV_TARGET]
X_val, y_val = val_df[feature_cols], val_df[REV_TARGET]

print(f"Train rows: {len(X_train):,} | Val rows: {len(X_val):,} | Features: {len(feature_cols)}")

# Train & benchmark each tree model
models = get_tree_models(cat_cols, num_cols)
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")  
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    metrics = evaluate_predictions(y_val, val_preds)
    metrics["Model"] = name
    results.append(metrics)

# Compare validation scores side-by-side
results_df = pd.DataFrame(results)[["Model", "RMSE_log1p", "R2_log1p", "MAE_raw", "RMSE_raw"]]
print("\n--- Validation Benchmark ---")
print(results_df.to_string(index=False))


Train rows: 736,117 | Val rows: 205,428 | Features: 47

Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6495
[LightGBM] [Info] Number of data points in the train set: 736117, number of used features: 47
[LightGBM] [Info] Start training from score 12.290239

Training XGBoost...

Training HistGradientBoosting...

Training RandomForest...

--- Validation Benchmark ---
               Model  RMSE_log1p  R2_log1p   MAE_raw  RMSE_raw
            LightGBM      0.6213    0.8639 234904.69 848032.10
             XGBoost      0.6177    0.8655 233512.44 831891.11
HistGradientBoosting      0.6216    0.8637 237409.03 861628.49
        RandomForest      0.6265    0.8616 215624.92 675260.46


In [ ]:
import json
import joblib

full_spine.to_parquet("full_spine_ready.parquet", index=False)



# Save feature lists & ensemble weight as plain JSON
metadata = {
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
}
with open("airline_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)




Exported Parquet + native XGBoost JSON + RF joblib + metadata JSON cleanly!
